In [44]:
import pandas as pd

# Charger le dataset
df = pd.read_csv("../datasets/custom_hp_dataset.csv")

# Afficher les premières lignes
df.head(10)

,text,category
0,My HP printer cannot connect to the WiFi netwo...,network
1,The laptop keeps disconnecting from the intern...,network
2,I cannot connect my HP printer to the wireless...,network
3,The WiFi connection on my computer is unstable...,network
4,The Ethernet connection is not detected on my ...,network
5,The printer says network unavailable although ...,network
6,I am unable to access the internet after updat...,network
7,The wireless printer cannot communicate with t...,network
8,My HP laptop cannot find any available WiFi ne...,network
9,The internet speed becomes extremely slow when...,network


In [45]:
df.columns

Index(['text', 'category'], dtype='str')

In [47]:
df["category"].value_counts()

category
network          20
hardware         20
software         20
configuration    20
performance      20
warranty         20
delivery         20
billing          20
inquiry          20
Name: count, dtype: int64

In [48]:
df["category"].value_counts().head(30)

category
network          20
hardware         20
software         20
configuration    20
performance      20
warranty         20
delivery         20
billing          20
inquiry          20
Name: count, dtype: int64

In [28]:
# Création mapping catégories

category_mapping = {
    "Hardware issue": "hardware",
    "Display issue": "hardware",
    "Battery life": "hardware",

    "Software bug": "software",
    "Account access": "software",

    "Network problem": "network",
    "Peripheral compatibility": "network",

    "Product setup": "configuration",
    "Installation support": "configuration",
    "Product compatibility": "configuration",

    "Delivery problem": "delivery",

    "Refund request": "warranty",
    "Cancellation request": "warranty",

    "Payment issue": "billing",

    "Data loss": "performance",

    "Product recommendation": "inquiry"
}

# Nouvelle colonne
df["problem_category"] = df["Ticket Subject"].map(category_mapping)

# Vérification
df[["Ticket Subject", "problem_category"]].head()

,Ticket Subject,problem_category
0,Product setup,configuration
1,Peripheral compatibility,network
2,Network problem,network
3,Account access,software
4,Data loss,performance


In [29]:
df["problem_category"].value_counts()

problem_category
configuration    1626
hardware         1567
software         1083
warranty         1063
network          1035
delivery          561
billing           526
inquiry           517
performance       491
Name: count, dtype: int64

In [30]:
# Combiner sujet + description

df["text"] = (
    df["Ticket Subject"].fillna('') + " " +
    df["Ticket Description"].fillna('')
)

# Dataset final
ml_df = df[["text", "problem_category"]]

# Vérification
ml_df.head()

,text,problem_category
0,Product setup I'm having an issue with the {pr...,configuration
1,Peripheral compatibility I'm having an issue w...,network
2,Network problem I'm facing a problem with my {...,network
3,Account access I'm having an issue with the {p...,software
4,Data loss I'm having an issue with the {produc...,performance


In [31]:
ml_df.isnull().sum()

text                0
problem_category    0
dtype: int64

In [32]:
ml_df = ml_df.dropna()

print(ml_df.shape)

(8469, 2)


In [33]:
# Features et labels

X = ml_df["text"]
y = ml_df["problem_category"]

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 6775
Test size: 1694


In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)

(6775, 5000)


In [36]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

print("Model trained successfully!")

Model trained successfully!


In [37]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9976387249114522

Classification Report:

               precision    recall  f1-score   support

      billing       1.00      1.00      1.00       105
configuration       0.99      1.00      1.00       337
     delivery       1.00      1.00      1.00       113
     hardware       1.00      1.00      1.00       334
      inquiry       1.00      1.00      1.00       101
      network       1.00      0.99      1.00       190
  performance       0.98      1.00      0.99       102
     software       1.00      1.00      1.00       216
     warranty       1.00      0.99      1.00       196

     accuracy                           1.00      1694
    macro avg       1.00      1.00      1.00      1694
 weighted avg       1.00      1.00      1.00      1694



In [38]:
import joblib

# Sauvegarder modèle
joblib.dump(model, "../models/complaint_classifier.pkl")

# Sauvegarder vectorizer
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


In [42]:
# Test manuel

sample = [
    "le wifi ne fonctionne plus"
]

sample_tfidf = vectorizer.transform(sample)

prediction = model.predict(sample_tfidf)

print("Predicted category:", prediction[0])

Predicted category: configuration
